In [12]:
pip install sentence-transformers faiss-cpu
pip install -U sentence-transformers

SyntaxError: invalid syntax (929854719.py, line 1)

In [2]:
from pathlib import Path
import re
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss


# ============================================================
# 1. LOAD POLICY DOCUMENTS
# ============================================================

POLICY_DIR = Path("../data/company_policy")

policy_files = sorted(POLICY_DIR.glob("*.txt"))

print(f"Found {len(policy_files)} policy documents")


documents = []

for file_path in policy_files:

    content = file_path.read_text(encoding="utf-8")

    if not content.strip():
        print(f"Skipping empty file: {file_path.name}")
        continue

    documents.append({
        "name": file_path.name,
        "content": content
    })


print(f"Loaded {len(documents)} documents")


# ============================================================
# 2. CLEAN TEXT
# ============================================================

def clean_text(text):

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    text = re.sub(r"[ \t]+", " ", text)

    text = re.sub(r"\n{3,}", "\n\n", text)

    lines = [line.strip() for line in text.splitlines()]

    return "\n".join(lines).strip()


# ============================================================
# 3. METADATA
# ============================================================

def get_metadata(filename):

    if "travel_policy_india" in filename:
        return {
            "source": filename,
            "policy_type": "travel",
            "country": "India"
        }

    elif "travel_policy_us" in filename:
        return {
            "source": filename,
            "policy_type": "travel",
            "country": "US"
        }

    elif "airport_policy" in filename:
        return {
            "source": filename,
            "policy_type": "airport",
            "country": "Global"
        }

    elif "employee_eligibility" in filename:
        return {
            "source": filename,
            "policy_type": "eligibility",
            "country": "Global"
        }

    elif "expense_policy" in filename:
        return {
            "source": filename,
            "policy_type": "expense",
            "country": "Global"
        }

    elif "cancellation_policy" in filename:
        return {
            "source": filename,
            "policy_type": "cancellation",
            "country": "Global"
        }

    elif "approval_policy" in filename:
        return {
            "source": filename,
            "policy_type": "approval",
            "country": "Global"
        }

    return {
        "source": filename,
        "policy_type": "unknown",
        "country": "unknown"
    }


# ============================================================
# 4. CHUNK DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)


chunks = []

for doc in documents:

    cleaned = clean_text(doc["content"])

    metadata = get_metadata(doc["name"])

    split_texts = text_splitter.split_text(cleaned)

    for chunk_id, chunk_text in enumerate(split_texts):

        chunks.append({
            "text": chunk_text,
            "metadata": {
                **metadata,
                "chunk_id": chunk_id
            }
        })


print(f"Created {len(chunks)} chunks")

Found 7 policy documents
Loaded 7 documents
Created 28 chunks


In [13]:
# ============================================================
# STRONGER EMBEDDING MODEL
# ============================================================

from sentence_transformers import SentenceTransformer
import faiss

model = SentenceTransformer("BAAI/bge-large-en-v1.5")

texts = [chunk["text"] for chunk in chunks]

# Document embeddings
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Number of chunks:", len(texts))
print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of chunks: 28
Embedding shape: (28, 1024)


In [4]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


In [14]:
# ============================================================
# BUILD FAISS INDEX
# ============================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", dimension)

FAISS index created
Number of vectors: 28
Vector dimension: 1024


In [15]:
# ============================================================
# SEMANTIC SEARCH
# ============================================================

def search_policy(query, top_k=3):

    # BGE v1.5 uses a retrieval instruction for queries
    query_text = (
        "Represent this sentence for searching relevant passages: "
        + query
    )

    query_embedding = model.encode(
        [query_text],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = chunks[idx]

        results.append({
            "score": float(score),
            "text": chunk["text"],
            "metadata": chunk["metadata"]
        })

    return results

In [17]:
query = "Can I use Uber for an airport trip late at night?"

results = search_policy(query, top_k=5)

for i, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"Result {i}")
    print(f"Similarity Score: {result['score']:.4f}")
    print(f"Source: {result['metadata']['source']}")
    print(f"Policy Type: {result['metadata']['policy_type']}")
    print(f"Country: {result['metadata']['country']}")
    print("\nText:")
    print(result["text"])

Result 1
Similarity Score: 0.6737
Source: travel_policy_us.txt
Policy Type: travel
Country: US

Text:
5. Late-Night Travel
Business travel between 10:00 PM and 6:00 AM is permitted when it is related to an approved business activity. Additional approval may be required when the trip exceeds the applicable spending limit.

6. Required Information
Employees should provide a valid employee ID, business purpose, trip date, and cost when submitting an expense.
Result 2
Similarity Score: 0.6737
Source: travel_policy_india.txt
Policy Type: travel
Country: India

Text:
5. Late-Night Travel
Business travel between 10:00 PM and 6:00 AM is permitted when it is related to an approved business activity. Additional approval may be required when the trip exceeds the applicable spending limit.

6. Required Information
Employees should provide a valid employee ID, business purpose, trip date, and cost when submitting an expense.
Result 3
Similarity Score: 0.6333
Source: airport_policy.txt
Policy Type: 

In [18]:
# ============================================================
# DAY 2 - COMPLETE SEMANTIC SEARCH + RERANKING PIPELINE
# ============================================================

from pathlib import Path
import re
import numpy as np
import faiss

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder


# ============================================================
# 1. LOAD POLICY DOCUMENTS
# ============================================================

POLICY_DIR = Path("../data/company_policy")

policy_files = sorted(POLICY_DIR.glob("*.txt"))

documents = []

for file_path in policy_files:

    try:
        content = file_path.read_text(encoding="utf-8")

        if not content.strip():
            print(f"Skipping empty document: {file_path.name}")
            continue

        documents.append({
            "name": file_path.name,
            "content": content
        })

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")


print("Documents loaded:", len(documents))

for doc in documents:
    print("-", doc["name"])


# ============================================================
# 2. CLEAN DOCUMENTS
# ============================================================

def clean_text(text):

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Replace tabs with spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


for doc in documents:
    doc["content"] = clean_text(doc["content"])


# ============================================================
# 3. ADD METADATA
# ============================================================

def get_metadata(filename):

    filename_lower = filename.lower()

    if "travel_policy_india" in filename_lower:
        return "travel", "India"

    elif "travel_policy_us" in filename_lower:
        return "travel", "US"

    elif "airport_policy" in filename_lower:
        return "airport", "Global"

    elif "employee_eligibility" in filename_lower:
        return "eligibility", "Global"

    elif "expense_policy" in filename_lower:
        return "expense", "Global"

    elif "cancellation_policy" in filename_lower:
        return "cancellation", "Global"

    elif "approval_policy" in filename_lower:
        return "approval", "Global"

    else:
        return "unknown", "Global"


# ============================================================
# 4. CHUNK DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for doc in documents:

    policy_type, country = get_metadata(doc["name"])

    doc_chunks = splitter.split_text(doc["content"])

    for i, chunk_text in enumerate(doc_chunks):

        chunks.append({
            "text": chunk_text,
            "metadata": {
                "source": doc["name"],
                "policy_type": policy_type,
                "country": country,
                "chunk_id": i
            }
        })


print("\nTotal chunks:", len(chunks))


# ============================================================
# 5. LOAD STRONGER EMBEDDING MODEL
# ============================================================

print("\nLoading BGE-large embedding model...")

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5"
)

texts = [chunk["text"] for chunk in chunks]


# ============================================================
# 6. GENERATE EMBEDDINGS
# ============================================================

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("\nEmbedding shape:", embeddings.shape)


# ============================================================
# 7. BUILD FAISS VECTOR STORE
# ============================================================

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("\nFAISS index created")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", dimension)


# ============================================================
# 8. SEMANTIC SEARCH
# ============================================================

def search_policy(query, top_k=5):

    # BGE v1.5 query instruction
    query_text = (
        "Represent this sentence for searching relevant passages: "
        + query
    )

    query_embedding = model.encode(
        [query_text],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        chunk = chunks[idx]

        results.append({
            "score": float(score),
            "text": chunk["text"],
            "metadata": chunk["metadata"]
        })

    return results


# ============================================================
# 9. LOAD RERANKER
# ============================================================

print("\nLoading BGE reranker...")

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

print("Reranker loaded successfully.")


# ============================================================
# 10. RERANK RETRIEVED RESULTS
# ============================================================

def rerank_results(query, results, top_k=3):

    pairs = []

    for result in results:

        pairs.append([
            query,
            result["text"]
        ])

    rerank_scores = reranker.predict(pairs)

    for result, score in zip(results, rerank_scores):

        result["rerank_score"] = float(score)

    # Highest reranker score first
    results = sorted(
        results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return results[:top_k]


# ============================================================
# 11. TEST QUERY
# ============================================================

query = "Can I use Uber for an airport trip late at night?"


# First retrieve 5 candidates using FAISS
initial_results = search_policy(
    query,
    top_k=5
)


# Then rerank those 5 candidates
final_results = rerank_results(
    query,
    initial_results,
    top_k=3
)


# ============================================================
# 12. DISPLAY INITIAL FAISS RESULTS
# ============================================================

print("\n")
print("=" * 80)
print("INITIAL FAISS TOP 5 RESULTS")
print("=" * 80)

for i, result in enumerate(initial_results, 1):

    print(f"\nResult {i}")
    print("-" * 80)

    print(
        f"FAISS Similarity Score: "
        f"{result['score']:.4f}"
    )

    print(
        f"Source: "
        f"{result['metadata']['source']}"
    )

    print(
        f"Policy Type: "
        f"{result['metadata']['policy_type']}"
    )

    print(
        f"Country: "
        f"{result['metadata']['country']}"
    )

    print(
        f"Chunk ID: "
        f"{result['metadata']['chunk_id']}"
    )

    print("\nText:")
    print(result["text"])


# ============================================================
# 13. DISPLAY FINAL RERANKED RESULTS
# ============================================================

print("\n\n")
print("=" * 80)
print("FINAL RERANKED TOP 3 RESULTS")
print("=" * 80)

for i, result in enumerate(final_results, 1):

    print(f"\nResult {i}")
    print("-" * 80)

    print(
        f"Rerank Score: "
        f"{result['rerank_score']:.4f}"
    )

    print(
        f"FAISS Similarity Score: "
        f"{result['score']:.4f}"
    )

    print(
        f"Source: "
        f"{result['metadata']['source']}"
    )

    print(
        f"Policy Type: "
        f"{result['metadata']['policy_type']}"
    )

    print(
        f"Country: "
        f"{result['metadata']['country']}"
    )

    print(
        f"Chunk ID: "
        f"{result['metadata']['chunk_id']}"
    )

    print("\nText:")
    print(result["text"])


# ============================================================
# 14. SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)

print("Documents:", len(documents))
print("Chunks:", len(chunks))
print("Embedding model: BAAI/bge-large-en-v1.5")
print("Embedding dimension:", dimension)
print("Vector store: FAISS")
print("Initial retrieval: Top 5")
print("Reranker: BAAI/bge-reranker-base")
print("Final retrieval: Top 3")

print("\nDay 2 semantic retrieval pipeline completed.")

Documents loaded: 7
- airport_policy.txt
- approval_policy.txt
- cancellation_policy.txt
- employee_eligibility.txt
- expense_policy.txt
- travel_policy_india.txt
- travel_policy_us.txt

Total chunks: 28

Loading BGE-large embedding model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding shape: (28, 1024)

FAISS index created
Number of vectors: 28
Vector dimension: 1024

Loading BGE reranker...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Reranker loaded successfully.


INITIAL FAISS TOP 5 RESULTS

Result 1
--------------------------------------------------------------------------------
FAISS Similarity Score: 0.6737
Source: travel_policy_us.txt
Policy Type: travel
Country: US
Chunk ID: 3

Text:
5. Late-Night Travel
Business travel between 10:00 PM and 6:00 AM is permitted when it is related to an approved business activity. Additional approval may be required when the trip exceeds the applicable spending limit.

6. Required Information
Employees should provide a valid employee ID, business purpose, trip date, and cost when submitting an expense.

Result 2
--------------------------------------------------------------------------------
FAISS Similarity Score: 0.6737
Source: travel_policy_india.txt
Policy Type: travel
Country: India
Chunk ID: 3

Text:
5. Late-Night Travel
Business travel between 10:00 PM and 6:00 AM is permitted when it is related to an approved business activity. Additional approval may be required when

In [24]:
# ============================================================
# DAY 2 - RAG PIPELINE WITH OLLAMA
# ============================================================

from pathlib import Path
import subprocess
import requests


# ============================================================
# 1. FIND AVAILABLE OLLAMA MODEL
# ============================================================

def get_ollama_model():

    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            check=True
        )

        lines = result.stdout.strip().split("\n")

        if len(lines) <= 1:
            raise Exception("No Ollama models found.")

        models = []

        for line in lines[1:]:
            if line.strip():
                model_name = line.split()[0]
                models.append(model_name)

        # Prefer common chat models
        preferred_keywords = [
            "llama",
            "qwen",
            "mistral",
            "gemma"
        ]

        for keyword in preferred_keywords:
            for model in models:
                if keyword in model.lower():
                    return model

        # Otherwise use the first available model
        return models[0]

    except Exception as e:
        print("Could not find Ollama model.")
        print("Error:", e)
        return None


OLLAMA_MODEL = get_ollama_model()

print("=" * 70)
print("OLLAMA MODEL")
print("=" * 70)
print("Using model:", OLLAMA_MODEL)


# ============================================================
# 2. CREATE RAG CONTEXT
# ============================================================

def build_context(results):

    context_parts = []

    for i, result in enumerate(results, 1):

        metadata = result["metadata"]

        source = metadata["source"]
        country = metadata["country"]
        policy_type = metadata["policy_type"]

        context_parts.append(
            f"""
SOURCE {i}
Source File: {source}
Country: {country}
Policy Type: {policy_type}

Policy Content:
{result["text"]}
"""
        )

    return "\n".join(context_parts)


# ============================================================
# 3. CALL OLLAMA
# ============================================================

def ask_ollama(prompt):

    try:

        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )

        response.raise_for_status()

        data = response.json()

        return data["response"].strip()

    except Exception as e:

        return f"Ollama Error: {e}"


# ============================================================
# IMPROVED DAY 2 RAG PIPELINE
# Uses Top 5 retrieved + reranked chunks as LLM context
# ============================================================

# ============================================================
# FINAL RAG FUNCTION - BETTER CONTEXT HANDLING
# ============================================================

# ============================================================
# SIMPLE RAG PIPELINE
# ============================================================

# ============================================================
# RAG ANSWER - FINAL VERSION FOR LLAMA 3.2
# ============================================================

def rag_answer(query):

    # --------------------------------------------------------
    # 1. Retrieve relevant policy chunks
    # --------------------------------------------------------

    results = search_policy(query, top_k=5)

    # --------------------------------------------------------
    # 2. Build context
    # --------------------------------------------------------

    context = ""

    for i, r in enumerate(results, 1):
        context += f"""
================ POLICY CHUNK {i} ================
Source: {r['metadata']['source']}
Country: {r['metadata']['country']}
Policy Type: {r['metadata']['policy_type']}

{r['text']}

"""

    # --------------------------------------------------------
    # 3. Prompt
    # --------------------------------------------------------

    prompt = f"""
You are an AI assistant for a company's travel and expense
policy.

Your job is to answer the employee's question using the
POLICY CONTEXT provided below.

IMPORTANT RULES:

1. The POLICY CONTEXT is the source of truth.
2. Read ALL policy chunks before answering.
3. If ANY policy chunk contains information that answers
   the question, USE THAT INFORMATION.
4. Do not say "I could not find information" if the answer
   is present anywhere in the policy context.
5. Do not use outside knowledge.
6. Do not invent rules, limits, approvals, or exceptions.
7. If multiple chunks are relevant, combine them carefully.
8. If the question is about airport travel, use airport
   policy information when available.
9. If the question is about India, prefer India-specific
   policy information.
10. If the question is about the US, prefer US-specific
    policy information.
11. Always mention the source file used.

If the requested information genuinely does NOT exist in
ANY of the policy chunks, answer exactly:

"I could not find information about this in the available
policy documents."

VERY IMPORTANT:
Do not confuse "the information is in another chunk" with
"the information is unavailable."

For example, if one chunk says:

"Airport trips between 10:00 PM and 6:00 AM are allowed
when connected to an approved business journey."

and the employee asks:

"Can I use Uber for an airport trip late at night?"

the correct answer is YES, because the policy explicitly
contains the relevant rule.

Now answer the employee's question.

================ POLICY CONTEXT ================

{context}

================ EMPLOYEE QUESTION ================

{query}

================ ANSWER ================

Answer directly and concisely.
"""

    # --------------------------------------------------------
    # 4. Send to Ollama
    # --------------------------------------------------------

    answer = ask_ollama(prompt)

    # --------------------------------------------------------
    # 5. Collect unique sources
    # --------------------------------------------------------

    sources = []

    for r in results:
        source = r["metadata"]["source"]

        if source not in sources:
            sources.append(source)

    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "retrieved_results": results
    }


# ============================================================
# TEST 1 - AIRPORT LATE NIGHT
# ============================================================

result = rag_answer(
    "Can I use Uber for an airport trip late at night?"
)

print("=" * 80)
print("RAG TEST 1 - AIRPORT LATE-NIGHT")
print("=" * 80)

print("\nQuestion:")
print(result["query"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print("-", source)


# ============================================================
# TEST 2 - HALLUCINATION
# ============================================================

result = rag_answer(
    "Does the company reimburse hotel expenses?"
)

print("\n")
print("=" * 80)
print("HALLUCINATION TEST")
print("=" * 80)

print("\nQuestion:")
print(result["query"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print("-", source)


# ============================================================
# TEST 3 - INDIA TRAVEL LIMIT
# ============================================================

result = rag_answer(
    "What is the standard individual business ride limit in India?"
)

print("\n")
print("=" * 80)
print("RAG TEST 2 - INDIA TRAVEL LIMIT")
print("=" * 80)

print("\nQuestion:")
print(result["query"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print("-", source)

    
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("DAY 2 RAG PIPELINE")
print("=" * 80)

print("""
User Question
      ↓
BGE-large Embedding
      ↓
FAISS Semantic Search
      ↓
Top 5 Candidates
      ↓
BGE Reranking
      ↓
Top 5 Reranked Context
      ↓
Strict RAG Prompt
      ↓
Ollama LLM
      ↓
Answer + Sources
""")

print("Embedding Model :", "BAAI/bge-large-en-v1.5")
print("Vector Store    :", "FAISS")
print("Reranker        :", "BAAI/bge-reranker-base")
print("LLM             :", OLLAMA_MODEL)

OLLAMA MODEL
Using model: llama3.2:3b
RAG TEST 1 - AIRPORT LATE-NIGHT

Question:
Can I use Uber for an airport trip late at night?

Answer:
Source: airport_policy.txt

Yes, airport trips between 10:00 PM and 6:00 AM are allowed when connected to an approved business journey (Policy Chunk 4, Section 4).

Sources:
- travel_policy_us.txt
- travel_policy_india.txt
- airport_policy.txt


HALLUCINATION TEST

Question:
Does the company reimburse hotel expenses?

Answer:
I could not find information about this in the available policy documents.

Sources:
- airport_policy.txt
- expense_policy.txt
- cancellation_policy.txt
- travel_policy_us.txt


RAG TEST 2 - INDIA TRAVEL LIMIT

Question:
What is the standard individual business ride limit in India?

Answer:
According to Policy CHUNK 1 (Source: travel_policy_india.txt) and Policy CHUNK 3 (Source: expense_policy.txt), the standard reimbursement limit for an individual business ride in India is INR 2,000 per trip.

Sources:
- travel_policy_india.